In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
import os
os.chdir("/content/gdrive/MyDrive/LLM/CBLLMMODEL")

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
Hawkishnes_Df = pd.read_excel("CBRT_DecisionReportsV3.xlsx", index_col="Unnamed: 0")
print(len(Hawkishnes_Df))
Hawkishnes_Df.head()

In [ ]:
GovernorInterestDf = pd.read_excel("Dataset/GovernorInterestRate.xlsx", index_col = "Date")
print(len(GovernorInterestDf))
GovernorInterestDf.head(5)

In [ ]:
index_list = list(GovernorInterestDf.index)

for row_index in range(1, len(GovernorInterestDf)):

    formerindex = index_list[row_index-1]
    currentindex = index_list[row_index]


    formerinterestrate = GovernorInterestDf.loc[formerindex, "Lend"]
    currentinterestrate = GovernorInterestDf.loc[currentindex, "Lend"]
    formerdecision1 = GovernorInterestDf.loc[formerindex, "Decision1"]
    formerdecision2 = GovernorInterestDf.loc[formerindex, "Decision2"]

    if currentinterestrate > formerinterestrate:
        GovernorInterestDf.loc[currentindex, "Decision1"] = "Raise"
        GovernorInterestDf.loc[currentindex, "Decision2"] = "Raise"

    elif currentinterestrate < formerinterestrate:
        GovernorInterestDf.loc[currentindex, "Decision1"] = "Cut"
        GovernorInterestDf.loc[currentindex, "Decision2"] = "Cut"
    else:
        GovernorInterestDf.loc[currentindex, "Decision2"] = "Unchanged"

        if formerdecision1 == "Cut" or formerdecision1 == "Cut_Unchanged":
            GovernorInterestDf.loc[currentindex, "Decision1"] = "Cut_Unchanged"
        else:
            GovernorInterestDf.loc[currentindex, "Decision1"] = "Raise_Unchanged"


GovernorInterestDf["Decision3"] = ["Cut" if x == "Cut" or x == "Cut_Unchanged" else "Raise" for x in GovernorInterestDf.Decision1]




GovernorInterestDf.head(5)

In [ ]:
GovernorInterestDf.Decision3.value_counts()

In [ ]:
RegimeDf = pd.read_excel("Dataset/RegimeDf.xlsx", index_col ="Unnamed: 0" )
RegimeDf.set_index(GovernorInterestDf.index, inplace=True)
print(len(RegimeDf))
RegimeDf.head(2)

In [ ]:
GovernorInterestRegimeDf = pd.concat((RegimeDf["Regime"], GovernorInterestDf[["Decision1","Decision2","Decision3", "Governor"]]) , axis = 1)
GovernorInterestRegimeDf.head()

In [ ]:
Hawkishnes_Df.head(2)

In [ ]:
Sentence_Df = Hawkishnes_Df.copy()
Sentence_Df[["Regime", "Decision1","Decision2","Decision3", "Governor"]] = None
Sentence_Df.head()

for row in Sentence_Df.index:
    date = Sentence_Df.loc[row, "Date"]
    values = list(GovernorInterestRegimeDf.loc[date])
    Sentence_Df.loc[row, ["Regime", "Decision1","Decision2","Decision3", "Governor"]] = values

Sentence_Df.head(20)


In [ ]:
Sentence_Df.to_excel("CBRT_SentencesDf.xlsx")

In [ ]:
ReportsDf = pd.DataFrame(index = GovernorInterestRegimeDf.index,
                        columns = ["Content", "Neutral", "Hawkish", "Dovish", "negative", "positive", "Central Bank",
                                   "Firms", "Households", "Financial Sector", "Government"] )

ReportsDf.head(2)

In [ ]:
for row in ReportsDf.index:

    Aux_df = Sentence_Df[Hawkishnes_Df["Date"] == row]
    content = ""
    for sentence in Aux_df.Sentences:
        content += sentence

    Neutral = list(Aux_df["Hawkish/Dovish"]).count("Neutral")
    Hawkish = list(Aux_df["Hawkish/Dovish"]).count("Hawkish")
    Dovish = list(Aux_df["Hawkish/Dovish"]).count("Dovish")

    negative = list(Aux_df["Sentiment"]).count("negative")
    positive = list(Aux_df["Sentiment"]).count("positive")

    CentralBank = list(Aux_df["Agent"]).count("Central Bank")
    Firms = list(Aux_df["Agent"]).count("Firms")
    Households = list(Aux_df["Agent"]).count("Households")
    FinancialSector = list(Aux_df["Agent"]).count("Financial Sector")
    Government = list(Aux_df["Agent"]).count("Government")

    ReportsDf.loc[row, "Content"] = content
    ReportsDf.loc[row, "Neutral"] = Neutral
    ReportsDf.loc[row, "Hawkish"] = Hawkish
    ReportsDf.loc[row, "Dovish"] = Dovish
    ReportsDf.loc[row, "negative"] = negative
    ReportsDf.loc[row, "positive"] = positive
    ReportsDf.loc[row, "Central Bank"] = CentralBank
    ReportsDf.loc[row, "Firms"] = Firms
    ReportsDf.loc[row, "Households"] = Households
    ReportsDf.loc[row, "Financial Sector"] = FinancialSector
    ReportsDf.loc[row, "Government"] = Government


In [ ]:
ReportsDf.head(2)

In [ ]:
ReportsDf = pd.concat((ReportsDf, GovernorInterestRegimeDf), axis = 1)
ReportsDf.head(3)

In [ ]:
ReportsDf["hawkishness"] = [i / (i+j+k) for i,j,k in zip(ReportsDf.Hawkish, ReportsDf.Dovish, ReportsDf.Neutral)]
ReportsDf["marketness"] = [(i+j+k+z) / (i+j+k+z+p) for i,j,k,z,p in zip(ReportsDf.Firms, ReportsDf.Households, ReportsDf["Financial Sector"], ReportsDf.Government,ReportsDf["Central Bank"])]
ReportsDf["hawkish"] = ["YES" if x > y else "NO" for x,y in zip(ReportsDf.Hawkish, ReportsDf.Dovish)]
# ReportsDf["Decision2"] = ["Cut" if x == "Cut" or x == "Cut_Unchanged" else "Raise" for x in ReportsDf.Decision1]
# ReportsDf["Decision3"] = ["Unchanged" if x == "Raise_Unchanged" or x == "Cut_Unchanged"else x for x in ReportsDf.Decision1]
ReportsDf["market"] = ["YES" if x >= 0.5 else "NO" for x in ReportsDf.marketness]
ReportsDf.head(10)

In [ ]:
ReportsDf["market"].value_counts()

In [ ]:
ReportsDf.to_excel("CBRT_ReportsDf.xlsx")
